In [98]:
import os
import pandas as pd

# Rutas
dir_entrada = r"M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN"
dir_salida = r"M:\material_isa\estadisticos\_Turberas\indices_analisis_NDVI"
os.makedirs(dir_salida, exist_ok=True)

# Procesar cada archivo
for archivo in os.listdir(dir_entrada):
    if archivo.endswith("_sin_periodo_nieve_filtr.csv"):
        nombre_turbera = archivo.replace("_sin_periodo_nieve_filtr.csv", "")
        ruta_csv = os.path.join(dir_entrada, archivo)

        try:
            df = pd.read_csv(ruta_csv, sep=';', encoding='latin1')

            # Convertir FECHA y NDVI
            df['FECHA'] = pd.to_datetime(df['FECHA'], errors='coerce')
            df['NDVI'] = pd.to_numeric(df['NDVI'], errors='coerce')
            df = df.dropna(subset=['FECHA', 'NDVI'])

            df['YEAR'] = df['FECHA'].dt.year

            resultados = []
            for año, grupo in df.groupby('YEAR'):
                grupo_ordenado = grupo.sort_values('FECHA')

                verdor_in_str = ""
                verdor_max_str = ""
                ndvi_max = ""
                periodo_verdor = ""

                if año != 2015:
                    verdor_in = grupo_ordenado.iloc[0]['FECHA']
                    verdor_in_str = verdor_in.strftime('%Y-%m-%d')

                if año != 2025:
                    idx_max = grupo_ordenado['NDVI'].idxmax()
                    verdor_max = grupo_ordenado.loc[idx_max, 'FECHA']
                    ndvi_max = grupo_ordenado.loc[idx_max, 'NDVI']
                    verdor_max_str = verdor_max.strftime('%Y-%m-%d')

                # Solo calcular periodo si ambas fechas existen
                if verdor_in_str and verdor_max_str:
                    periodo_verdor = (verdor_max - verdor_in).days

                media_ndvi = grupo_ordenado['NDVI'].mean()
                mediana_ndvi = grupo_ordenado['NDVI'].median()  # 👈 NUEVO

                resultados.append({
                    'YEAR': año,
                    'Verdor In': verdor_in_str,
                    'Verdor Max': verdor_max_str,
                    'NDVI Verdor Max': ndvi_max,
                    'Periodo Verdor': periodo_verdor,
                    'MEDIA NDVI': media_ndvi,
                    'MEDIANA NDVI': mediana_ndvi  # 👈 NUEVO
                })

            df_resultado = pd.DataFrame(resultados)
            salida_csv = os.path.join(dir_salida, f"{nombre_turbera}_analisis_NDVI.csv")
            df_resultado.to_csv(salida_csv, sep=';', index=False, encoding='utf-8')

            print(f"✅ Guardado: {salida_csv}")

        except Exception as e:
            print(f"❌ Error procesando {archivo}: {e}")

print("\n🎯 Proceso finalizado para todas las turberas.")


✅ Guardado: M:\material_isa\estadisticos\_Turberas\indices_analisis_NDVI\Bassa_Nera_analisis_NDVI.csv
✅ Guardado: M:\material_isa\estadisticos\_Turberas\indices_analisis_NDVI\Clots_de_Rialba_analisis_NDVI.csv
✅ Guardado: M:\material_isa\estadisticos\_Turberas\indices_analisis_NDVI\Estanyeres_analisis_NDVI.csv
✅ Guardado: M:\material_isa\estadisticos\_Turberas\indices_analisis_NDVI\Plan_deth_Horn_analisis_NDVI.csv
✅ Guardado: M:\material_isa\estadisticos\_Turberas\indices_analisis_NDVI\Pla_de_Sartari_analisis_NDVI.csv
✅ Guardado: M:\material_isa\estadisticos\_Turberas\indices_analisis_NDVI\Pleta_de_Molieres_analisis_NDVI.csv

🎯 Proceso finalizado para todas las turberas.
